# Violence Detection - Skeleton ST-GCN Fine-Tuning (Colab)

Trains a **binary violence / non-violence** classifier over human **skeletons**
(pose keypoints). Skeletons are *background-invariant* - this is the fix for the
previous models that false-fired on benign home backgrounds.

**Tuned for: free Colab T4 + a 4 GB local inference GPU.**
- Pose extraction uses `yolo11s-pose` (light, fast, 4 GB friendly). Keep the
  SAME model in the app's config for train/inference parity.
- `MAX_CLIPS_PER_CLASS` caps the dataset so Phase 1 finishes in one ~4h session.

**Run order:** mount Drive -> edit CONFIG cell -> Run all.
Phase 1 (skeleton extraction) is the only slow step and is **resume-safe**
(skips already-cached clips), so if Colab disconnects just re-run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =====================================================================
#  CONFIG  -  after you upload the whole Fine_tuning folder to
#  MyDrive/VisionAI/, these paths already point at this model folder.
#  (datasets live INSIDE it - see DATASETS.txt.)  "" = skip a source.
# =====================================================================
BASE = "/content/drive/MyDrive/VisionAI/Fine_tuning/01_violence_stgcn"

# RLVS  (PRIMARY)   layout: <RLVS>/Violence/*.mp4 , <RLVS>/NonViolence/*.mp4
RLVS_ROOT     = BASE + "/RLVS"

# RWF-2000 (OPTIONAL - official request only, not redistributable). "" = skip.
# layout: <root>/train/Fight, /train/NonFight, /val/Fight, /val/NonFight
RWF2000_ROOT  = ""

# Benign HOME negatives (your own benign clips; ALL treated as non_violence)
HOME_NEG_ROOT = BASE + "/home_negatives"

# Working dir: skeleton cache + trained checkpoint are written here
WORK_DIR      = BASE + "/_work"

# =====================================================================
#  HYPERPARAMETERS
# =====================================================================
POSE_MODEL    = "yolo11s-pose.pt"   # 4GB-friendly + fast. MUST match app config.
POSE_CONF     = 0.30
CLIP_LEN      = 32                   # frames sampled per clip
MAX_PERSONS   = 2                    # two-person interaction
IMG_NORM      = True                 # normalize coords to [-1,1] by frame size

# Caps to fit one ~4h Colab session. Set to 0 for "no cap" (full dataset).
MAX_CLIPS_PER_CLASS = 700            # per (dataset-source, label); raise if time allows

BATCH_SIZE    = 64                   # T4: 64; A100: 128-256
EPOCHS        = 60                   # max per fold; early stopping ends sooner
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
VAL_SPLIT     = 0.20
SEED          = 42

import os
os.makedirs(WORK_DIR, exist_ok=True)
CACHE_DIR = os.path.join(WORK_DIR, "skeleton_cache")
os.makedirs(CACHE_DIR, exist_ok=True)
print("Work dir :", WORK_DIR)
print("Cache dir:", CACHE_DIR)

In [ ]:
!pip -q install ultralytics tqdm
import torch, subprocess
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print(subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))

## Phase 1 - Extract skeletons (yolo11s-pose -> cached `.npy`)

Each clip becomes `[CLIP_LEN, MAX_PERSONS, 17, 3]` (T frames x up-to-2 people x
17 COCO joints x (x, y, conf)). Coords normalized by frame size so the *distance
between people* is preserved (a key violence cue). Resume-safe.

In [ ]:
import os, cv2, json, hashlib, numpy as np, random
from ultralytics import YOLO
from tqdm import tqdm

random.seed(SEED); np.random.seed(SEED)
USE_HALF = torch.cuda.is_available()
pose_model = YOLO(POSE_MODEL)
VIDEO_EXT = ('.mp4', '.avi', '.mov', '.mkv')

def _sample_indices(n_total, n):
    if n_total <= 0: return []
    if n_total >= n: return [int(v) for v in np.linspace(0, n_total - 1, n)]
    return list(range(n_total)) + [n_total - 1] * (n - n_total)

def extract_skeleton(video_path):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W = cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 1.0
    H = cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 1.0
    if total <= 0:
        cap.release(); return None
    idx_list = _sample_indices(total, CLIP_LEN)
    need = set(idx_list); grabbed = {}; fi = 0; maxneed = max(idx_list)
    while fi <= maxneed:
        ret, fr = cap.read()
        if not ret: break
        if fi in need: grabbed[fi] = fr
        fi += 1
    cap.release()
    if not grabbed: return None
    frames, last = [], None
    for fidx in idx_list:
        fr = grabbed.get(fidx, last)
        if fr is None: fr = next(iter(grabbed.values()))
        frames.append(fr); last = fr
    results = pose_model.predict(frames, conf=POSE_CONF, classes=[0], verbose=False, half=USE_HALF)
    clip = np.zeros((CLIP_LEN, MAX_PERSONS, 17, 3), dtype=np.float32)
    for t, res in enumerate(results):
        if res.keypoints is None or res.keypoints.data is None: continue
        kp = res.keypoints.data.cpu().numpy()
        if kp.shape[0] == 0: continue
        order = np.argsort(-kp[:, :, 2].mean(axis=1))[:MAX_PERSONS]
        for m, pidx in enumerate(order):
            person = kp[pidx].copy()
            if IMG_NORM:
                person[:, 0] = (person[:, 0] / W - 0.5) * 2.0
                person[:, 1] = (person[:, 1] / H - 0.5) * 2.0
            clip[t, m] = person
    return clip

def cache_path(video_path):
    h = hashlib.md5(video_path.encode()).hexdigest()[:16]
    base = os.path.splitext(os.path.basename(video_path))[0]
    return os.path.join(CACHE_DIR, base + "_" + h + ".npy")

def _list_videos(d):
    out = []
    for root, _, files in os.walk(d):
        for f in files:
            if f.lower().endswith(VIDEO_EXT):
                out.append(os.path.join(root, f))
    return out

def _cap(lst):
    random.shuffle(lst)
    return lst if MAX_CLIPS_PER_CLASS in (0, None) else lst[:MAX_CLIPS_PER_CLASS]

def gather_samples():
    samples = []   # (video_path, label, split)
    if RWF2000_ROOT:
        for split in ('train', 'val'):
            for sub, lab in (('Fight', 1), ('NonFight', 0)):
                d = os.path.join(RWF2000_ROOT, split, sub)
                if os.path.isdir(d):
                    for f in _cap(_list_videos(d)):
                        samples.append((f, lab, split))
    if RLVS_ROOT:
        for sub, lab in (('Violence', 1), ('NonViolence', 0)):
            d = os.path.join(RLVS_ROOT, sub)
            if os.path.isdir(d):
                for f in _cap(_list_videos(d)):
                    samples.append((f, lab, ''))
    if HOME_NEG_ROOT and os.path.isdir(HOME_NEG_ROOT):
        for f in _cap(_list_videos(HOME_NEG_ROOT)):
            samples.append((f, 0, ''))
    return samples

samples = gather_samples()
print("Source clips selected:", len(samples))
assert samples, "No clips found - check dataset paths in CONFIG."

manifest = []
for vpath, lab, split in tqdm(samples, desc="Extracting skeletons"):
    cpath = cache_path(vpath)
    if not os.path.exists(cpath):
        clip = extract_skeleton(vpath)
        if clip is None: continue
        np.save(cpath, clip)
    manifest.append({"npy": cpath, "label": lab, "split": split})

with open(os.path.join(WORK_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f)
print("Cached samples:", len(manifest))

## Phase 2 - Datasets / loaders (pool -> stratified 75/25)
Reads the tiny cached `.npy` skeletons (not video), so epochs are fast. All clips
are pooled and split 75/25 (stratified): the 75% feeds K-fold CV, the 25% is a
held-out test scored once at the end.

In [ ]:
import json, numpy as np, torch, random
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split

manifest = json.load(open(os.path.join(WORK_DIR, "manifest.json")))
random.seed(SEED); np.random.seed(SEED)

# Pool ALL cached clips, then a stratified 75/25 split into a train+val pool
# (where K-fold CV runs) and a held-out TEST set (touched once, in Evaluate).
all_items  = list(manifest)
all_labels = np.array([m["label"] for m in all_items])
trainval_items, test_items = train_test_split(
    all_items, test_size=0.25, random_state=SEED, stratify=all_labels)
print("Pool:", len(all_items), "| train+val:", len(trainval_items), " held-out test:", len(test_items))

class SkeletonDataset(Dataset):
    def __init__(self, items, augment=False):
        self.items = items; self.augment = augment
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        m = self.items[i]
        x = np.load(m["npy"]).astype(np.float32)
        if self.augment:
            if random.random() < 0.5: x[..., 0] *= -1.0
            x[..., :2] += np.random.normal(0, 0.01, x[..., :2].shape).astype(np.float32)
        x = torch.from_numpy(x).permute(3, 0, 2, 1).contiguous()   # -> (C, T, V, M)
        return x, m["label"]

def make_loaders(tr_items, va_items):
    tr_ds = SkeletonDataset(tr_items, augment=True)
    va_ds = SkeletonDataset(va_items, augment=False)
    labels = np.array([m["label"] for m in tr_items])
    cls_count = np.bincount(labels, minlength=2).astype(np.float32)
    cls_w = 1.0 / np.maximum(cls_count, 1.0)
    samp_w = [cls_w[l] for l in labels]
    sampler = WeightedRandomSampler(samp_w, num_samples=len(tr_items), replacement=True)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return tr_loader, va_loader, cls_w

_cnt = np.bincount(np.array([m["label"] for m in trainval_items]), minlength=2).astype(int)
print("Train-pool class counts:", dict(zip(["non_violence", "violence"], _cnt)))

## ST-GCN model (plain PyTorch, COCO-17 graph)
Spatial-temporal graph conv over the 17-joint COCO skeleton with a learnable
edge-importance mask. Multi-person handled by folding the person axis into the
batch and pooling at the end.

In [ ]:
import torch, torch.nn as nn, numpy as np
COCO_EDGES = [(0,1),(0,2),(1,3),(2,4),(0,5),(0,6),(5,6),(5,7),(7,9),
              (6,8),(8,10),(5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16)]
NUM_JOINTS = 17
def build_adjacency():
    A = np.zeros((NUM_JOINTS, NUM_JOINTS), dtype=np.float32)
    for i, j in COCO_EDGES: A[i, j] = 1.0; A[j, i] = 1.0
    A += np.eye(NUM_JOINTS, dtype=np.float32)
    deg = A.sum(1); Dinv = np.diag(1.0 / np.maximum(deg, 1e-6)).astype(np.float32)
    return (Dinv @ A).astype(np.float32)

class STGCNBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, residual=True):
        super().__init__()
        self.register_buffer("A", torch.tensor(A))
        self.edge_imp = nn.Parameter(torch.ones_like(self.A))
        self.gcn = nn.Conv2d(cin, cout, 1)
        self.tcn = nn.Sequential(nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, (9, 1), (stride, 1), (4, 0)), nn.BatchNorm2d(cout))
        if not residual: self.res = None
        elif cin == cout and stride == 1: self.res = nn.Identity()
        else: self.res = nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)), nn.BatchNorm2d(cout))
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        res = 0 if self.res is None else self.res(x)
        x = self.gcn(x)
        x = torch.einsum("nctv,vw->nctw", x, self.A * self.edge_imp)
        x = self.tcn(x)
        return self.relu(x + res)

class STGCN(nn.Module):
    def __init__(self, in_ch=3, num_classes=2):
        super().__init__()
        A = build_adjacency()
        self.data_bn = nn.BatchNorm1d(in_ch * NUM_JOINTS)
        self.layers = nn.ModuleList([
            STGCNBlock(in_ch, 64, A, residual=False), STGCNBlock(64, 64, A),
            STGCNBlock(64, 128, A, stride=2), STGCNBlock(128, 128, A),
            STGCNBlock(128, 256, A, stride=2), STGCNBlock(256, 256, A)])
        self.fc = nn.Linear(256, num_classes)
    def forward(self, x):
        N, C, T, V, M = x.shape
        x = x.permute(0, 4, 1, 3, 2).contiguous().view(N * M, C * V, T)
        x = self.data_bn(x)
        x = x.view(N * M, C, V, T).permute(0, 1, 3, 2).contiguous()
        for layer in self.layers: x = layer(x)
        x = x.mean(dim=[2, 3]).view(N, M, -1).mean(dim=1)
        return self.fc(x)

_m = STGCN(); _dummy = torch.zeros(2, 3, CLIP_LEN, NUM_JOINTS, MAX_PERSONS)
print("Model OK, output shape:", tuple(_m(_dummy).shape))

## Train - Stratified 5-Fold CV (best model by val_loss)
Stratified K-Fold on the 75% train+val pool keeps class balance in every fold.
Each fold trains with AdamW + cosine + AMP + class-weighted loss, checkpoints on
the **lowest val_loss** (smoother + earlier overfit signal than accuracy) and
**early-stops** on patience. The single best-val_loss fold is saved to
`WORK_DIR/violence_stgcn_best.pt`; the 25% held-out test is scored next.

In [ ]:
import os, copy, numpy as np, torch, torch.nn as nn
from sklearn.model_selection import StratifiedKFold
device = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ["non_violence", "violence"]
N_SPLITS = 5            # Stratified K-Fold on the 75% train+val pool
PATIENCE = 8            # early stop after this many epochs without val_loss gain
save_path = os.path.join(WORK_DIR, "violence_stgcn_best.pt")

def run_eval(model, loader, criterion):
    model.eval(); vloss = 0.0; vc = vn = 0; cm = np.zeros((2, 2), dtype=int)
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = criterion(out, y)
            vloss += loss.item() * y.size(0)
            p = out.argmax(1); vc += (p == y).sum().item(); vn += y.size(0)
            for t, pp in zip(y.cpu().numpy(), p.cpu().numpy()): cm[t, pp] += 1
    return vloss / max(vn, 1), vc / max(vn, 1), cm

def train_one_fold(tr_items, va_items, fold):
    tr_loader, va_loader, cls_w = make_loaders(tr_items, va_items)
    model = STGCN(in_ch=3, num_classes=2).to(device)
    w = torch.tensor(cls_w / cls_w.sum() * 2.0, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None
    best_vloss = float("inf"); best_va = 0.0; best_state = None; bad = 0
    for epoch in range(EPOCHS):
        model.train(); tc = tn = 0; run = 0.0
        for x, y in tr_loader:
            x, y = x.to(device), y.to(device); opt.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    out = model(x); loss = criterion(out, y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                out = model(x); loss = criterion(out, y); loss.backward(); opt.step()
            run += loss.item() * y.size(0)
            tc += (out.argmax(1) == y).sum().item(); tn += y.size(0)
        sched.step()
        vloss, va, _ = run_eval(model, va_loader, criterion)
        print("  [fold %d] ep %02d/%d  loss=%.4f train_acc=%.3f  val_loss=%.4f val_acc=%.3f" % (
            fold, epoch + 1, EPOCHS, run / max(tn, 1), tc / max(tn, 1), vloss, va))
        if vloss < best_vloss - 1e-4:
            best_vloss = vloss; best_va = va; best_state = copy.deepcopy(model.state_dict()); bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  [fold %d] early stop @ epoch %d (best val_loss=%.4f)" % (fold, epoch + 1, best_vloss))
                break
    return best_state, best_vloss, best_va

y_pool = np.array([m["label"] for m in trainval_items])
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_vloss, fold_vacc = [], []
global_best_vloss = float("inf"); global_best_state = None
for fold, (tr_idx, va_idx) in enumerate(skf.split(trainval_items, y_pool), 1):
    tr_items = [trainval_items[i] for i in tr_idx]
    va_items = [trainval_items[i] for i in va_idx]
    print("Fold %d/%d  train=%d  val=%d" % (fold, N_SPLITS, len(tr_items), len(va_items)))
    state, vloss, va = train_one_fold(tr_items, va_items, fold)
    fold_vloss.append(vloss); fold_vacc.append(va)
    if vloss < global_best_vloss:
        global_best_vloss = vloss; global_best_state = state
        print("  ** new global best val_loss=%.4f (fold %d) **" % (vloss, fold))

print("\nCV val_loss: %.4f +/- %.4f" % (float(np.mean(fold_vloss)), float(np.std(fold_vloss))))
print("CV val_acc : %.4f +/- %.4f" % (float(np.mean(fold_vacc)), float(np.std(fold_vacc))))
torch.save({"model_state_dict": global_best_state, "class_names": CLASS_NAMES,
    "val_loss": float(global_best_vloss),
    "cv_val_loss_mean": float(np.mean(fold_vloss)), "cv_val_acc_mean": float(np.mean(fold_vacc)),
    "config": {"clip_len": CLIP_LEN, "max_persons": MAX_PERSONS, "num_joints": NUM_JOINTS,
        "in_ch": 3, "img_norm": IMG_NORM, "coco_edges": COCO_EDGES,
        "pose_model": POSE_MODEL, "pose_conf": POSE_CONF}}, save_path)
print("Saved best-fold model (val_loss=%.4f) ->" % global_best_vloss, save_path)

## Evaluate - held-out 25% test (precision / recall for violence)
This is the honest generalization number: the test set was never seen during CV
or model selection. For home deployment you care most about **precision** (few
false alarms) - if it's low, raise the inference decision threshold.

In [ ]:
import numpy as np, torch
from torch.utils.data import DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(os.path.join(WORK_DIR, "violence_stgcn_best.pt"), map_location=device)
model = STGCN(in_ch=3, num_classes=2).to(device)
model.load_state_dict(ckpt["model_state_dict"]); model.eval()
test_loader = DataLoader(SkeletonDataset(test_items, augment=False),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
TP = FP = FN = TN = 0
with torch.no_grad():
    for x, y in test_loader:
        p = model(x.to(device)).argmax(1).cpu().numpy(); y = y.numpy()
        for t, pp in zip(y, p):
            if   t == 1 and pp == 1: TP += 1
            elif t == 0 and pp == 1: FP += 1
            elif t == 1 and pp == 0: FN += 1
            else: TN += 1
prec = TP / max(TP + FP, 1); rec = TP / max(TP + FN, 1)
f1 = 2 * prec * rec / max(prec + rec, 1e-9)
print("HELD-OUT TEST (25%)  -  n =", TP + FP + FN + TN)
print("Violence  precision=%.3f  recall=%.3f  f1=%.3f" % (prec, rec, f1))
print("Counts    TP=%d  FP=%d  FN=%d  TN=%d" % (TP, FP, FN, TN))
print("CV val_loss mean:", round(ckpt["cv_val_loss_mean"], 4), "| saved best val_loss:", round(ckpt["val_loss"], 4))

## Export & wire back
`violence_stgcn_best.pt` is in `WORK_DIR` on Drive. To use it:
1. Download it into the repo at `models/weights/violence_stgcn_best.pt`.
2. The checkpoint carries everything inference needs (`clip_len`, `max_persons`,
   `coco_edges`, `img_norm`, `pose_model`, `pose_conf`, `class_names`).
3. Ping me and I'll add a `ViolenceSTGCN` stage that reuses the YOLO11-pose
   keypoints already produced in `stages/indoor_action.py`, runs a sliding
   `CLIP_LEN` window, and emits a learned `violence` event (replacing the noisy
   `aggressive_guard` heuristic). Multimodal fusion in `core/pipeline.py` then
   corroborates with emotion + audio.

**Inference tip:** confidence threshold + temporal persistence before alerting,
same gating philosophy as the weapon stage.